# Summary Figures — Reservoir Computing on ADNI resting-state fMRI

Three publication-quality figures:
- **Fig 1** — Dataset overview (computed fresh)
- **Fig 2** — Reservoir model: loads `fig_lda.png` + FC / PCA panels
- **Fig 3** — Perturbation experiments: loads pre-saved PNGs from project root

In [1]:
import os, sys, warnings
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
from matplotlib.patches import Patch
from scipy.stats import gaussian_kde
warnings.filterwarnings("ignore")

# ── paths ──────────────────────────────────────────────────────────────────
ROOT    = "."                   # project root (where therapeutic_*.png lives)
TS_ROOT = "./timeseries"        # CN / AD / MCI sub-folders
OUT_DIR = "./summary_out"
os.makedirs(OUT_DIR, exist_ok=True)

# ── reproducibility ────────────────────────────────────────────────────────
RNG_SEED    = 42
N_CC_SAMPLE = 40      # random CN subjects to use
N_SITES     = 121     # parcels
TR          = 3.0
trial_duration = 139

COND_COLORS = {"CC": "#2196F3", "AD": "#E91E63"}
print("Imports OK")

Imports OK


In [2]:
rng = np.random.default_rng(RNG_SEED)

folder_to_label = {"CN": "CC", "AD": "AD"}
collected_signals, identifiers = [], []

for subfolder, label in folder_to_label.items():
    folder_path = os.path.join(TS_ROOT, subfolder)
    if not os.path.isdir(folder_path):
        print(f"WARNING: {folder_path} not found, skipping")
        continue
    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".npy"))
    if label == "CC":
        files = list(rng.choice(files, size=min(N_CC_SAMPLE, len(files)), replace=False))
        print(f"CC: randomly sampled {len(files)} files")
    else:
        print(f"AD: using all {len(files)} files")
    for fname in files:
        arr = np.load(os.path.join(folder_path, fname)).T   # → (T, N_sites)
        if arr.shape[1] == N_SITES and arr.shape[0] >= trial_duration:
            collected_signals.append(arr)
            patient_id = fname.split("_ses-")[0]
            identifiers.append([label, patient_id])

identifiers      = np.array(identifiers, dtype=object)
state_ID_numeric = [0 if row[0] == "CC" else 1 for row in identifiers]
patient_ID       = [row[1] for row in identifiers]

ctrl_indices = [i for i, c in enumerate(state_ID_numeric) if c == 0]
ad_indices   = [i for i, c in enumerate(state_ID_numeric) if c == 1]

# (N_sites, T) per subject — consistent with original notebook
concatenated_signals_list = [s.T for s in collected_signals]

print(f"CC: {len(ctrl_indices)}   AD: {len(ad_indices)}   Total: {len(state_ID_numeric)}")

CC: randomly sampled 40 files
AD: using all 151 files
CC: 40   AD: 145   Total: 185


In [3]:
# Stack all signals (T_total, N_sites) and compute covariance
all_signals = np.concatenate([s.T for s in concatenated_signals_list], axis=0)
centered    = all_signals - all_signals.mean(axis=0)
cov_mat     = np.cov(centered.T)           # (N_sites, N_sites)

evals, evecs = np.linalg.eigh(cov_mat)
order        = np.argsort(evals)[::-1]
sorted_eigenvalues_common  = evals[order]
sorted_eigenvectors_common = evecs[:, order]
expl_var = sorted_eigenvalues_common / sorted_eigenvalues_common.sum()

print(f"Top-5 explained variance: {expl_var[:5] * 100}")
print(f"Cov matrix shape: {cov_mat.shape}")

Top-5 explained variance: [22.68333386  6.80589265  5.64674974  4.5919734   3.36973269]
Cov matrix shape: (121, 121)


In [4]:
# Compute empirical FC per subject (pass-1: raw data)
FC_collected = []
N_PC_MODEL   = 50
ev50 = sorted_eigenvectors_common[:, :N_PC_MODEL]   # top-50 PCA eigenvectors

for sig in concatenated_signals_list:
    # PC-projected signal (same as original cell 22)
    pc_scores = sig.T @ ev50                # (T, 50)
    proj      = (pc_scores @ ev50.T).T      # (N_sites, T)
    fc        = np.nan_to_num(np.corrcoef(proj))
    FC_collected.append(fc)

fc_ctrl_mean = np.mean([FC_collected[i] for i in ctrl_indices], axis=0)
fc_ad_mean   = np.mean([FC_collected[i] for i in ad_indices],   axis=0)

print(f"FC matrices computed: {len(FC_collected)}")
print(f"Mean upper-tri FC  CC: {np.mean([FC_collected[i][np.triu_indices(N_SITES,k=1)].mean() for i in ctrl_indices]):.3f}")
print(f"Mean upper-tri FC  AD: {np.mean([FC_collected[i][np.triu_indices(N_SITES,k=1)].mean() for i in ad_indices]):.3f}")

FC matrices computed: 185
Mean upper-tri FC  CC: 0.199
Mean upper-tri FC  AD: 0.228


In [5]:
try:
    from nilearn import datasets as nl_datasets
    schaefer   = nl_datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)
    atlas_lbls = [l.decode() if isinstance(l, bytes) else l for l in schaefer.labels]
    NET_ORDER  = ["Vis","SomMot","DorsAttn","SalVentAttn","Limbic","Cont","Default"]
    NET_COLORS_MAP = dict(Vis="#1f77b4", SomMot="#ff7f0e", DorsAttn="#2ca02c",
                          SalVentAttn="#d62728", Limbic="#9467bd", Cont="#8c564b",
                          Default="#e377c2")
    def get_net(lbl):
        for n in NET_ORDER:
            if n in lbl: return n
        return "Subcortical"
    net_assign  = [get_net(l) for l in atlas_lbls]
    sort_key    = lambda i: (NET_ORDER.index(net_assign[i]) if net_assign[i] in NET_ORDER else 7, i)
    sorted_idx  = sorted(range(100), key=sort_key)
    sorted_nets = [net_assign[i] for i in sorted_idx]
    net_bounds  = [i - 0.5 for i in range(1, 100) if sorted_nets[i] != sorted_nets[i-1]]
    have_atlas  = True
    print("Atlas loaded")
except Exception as e:
    print(f"Atlas unavailable ({e}), using identity ordering")
    sorted_idx = list(range(100))
    net_bounds = []
    NET_COLORS_MAP = {}
    have_atlas = False

[fetch_atlas_schaefer_2018] Dataset found in C:\Users\user\nilearn_data\schaefer_2018

Atlas loaded


In [6]:
fig = plt.figure(figsize=(22, 13), facecolor="white")
fig.suptitle("Figure 1 — Dataset Overview", fontsize=15, fontweight="bold", y=0.99)
gs = gridspec.GridSpec(2, 4, figure=fig, hspace=0.42, wspace=0.38,
                       top=0.94, bottom=0.08)

# ── A: Acquisition table ──────────────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
ax_a.axis("off")
rows = [["Property", "Value"],
        ["Dataset",  "ADNI"],
        ["Software", "fMRIPrep 25.2.5"],
        ["TR",        "3.0 s"],
        ["Volumes",   "140 (~7 min)"],
        ["Atlas",     "Schaefer-100 +\nHO-21 sub-cortical"],
        ["Parcels",   "121 total"],
        ["Confounds", "24-HMP + WM + CSF"],
        ["Bandpass",  "0.01 – 0.10 Hz"],
        [f"CC (CN)",  f"{len(ctrl_indices)} sessions"],
        [f"AD",       f"{len(ad_indices)} sessions"]]
tbl = ax_a.table(cellText=rows[1:], colLabels=rows[0],
                 cellLoc="left", loc="center", bbox=[0, 0, 1, 1])
tbl.auto_set_font_size(False); tbl.set_fontsize(9)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor("#cccccc")
    if r == 0: cell.set_facecolor("#e0e0e0")
ax_a.set_title("(A) Acquisition Setup", fontsize=11, fontweight="bold", pad=6)

# ── B: Sample BOLD timeseries ────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1:3])
t = np.arange(trial_duration) * TR
parcels_show = [48, 0, 37, 103]
parcel_names = ["pCunPCC (Default)", "LH Vis-1", "Default Temp", "L-Thalamus"]
ex_cc = concatenated_signals_list[ctrl_indices[0]]
ex_ad = concatenated_signals_list[ad_indices[0]]
for row_i, (pi, pname) in enumerate(zip(parcels_show, parcel_names)):
    offset  = row_i * 5
    sig_cc_ = ex_cc[pi, :trial_duration]
    sig_ad_ = ex_ad[pi, :trial_duration]
    ax_b.plot(t, sig_cc_ + offset, lw=1.0, color="#2196F3", alpha=0.9,
              label="CC" if row_i == 0 else "")
    ax_b.plot(t, sig_ad_ + offset, lw=1.0, color="#E91E63", alpha=0.9,
              label="AD" if row_i == 0 else "")
    ax_b.text(-8, offset, pname, fontsize=7, va="center", ha="right")
ax_b.set_xlim(-15, t[-1] + 5)
ax_b.set_xlabel("Time (s)", fontsize=9)
ax_b.set_ylabel("BOLD + offset", fontsize=9)
ax_b.legend(fontsize=8, loc="upper right"); ax_b.tick_params(labelsize=7)
ax_b.set_yticks([])
ax_b.set_title("(B) Sample BOLD Timeseries", fontsize=11, fontweight="bold")

# ── C: Mean FC violin ────────────────────────────────────────────────────
ax_c = fig.add_subplot(gs[0, 3])
fc_means_cc = [FC_collected[i][np.triu_indices(N_SITES, k=1)].mean() for i in ctrl_indices]
fc_means_ad = [FC_collected[i][np.triu_indices(N_SITES, k=1)].mean() for i in ad_indices]
vp = ax_c.violinplot([fc_means_cc, fc_means_ad], positions=[0, 1], showmedians=True)
vp["bodies"][0].set_facecolor("#2196F3")
vp["bodies"][1].set_facecolor("#E91E63")
for k in ["cbars", "cmins", "cmaxes", "cmedians"]:
    vp[k].set_color("k"); vp[k].set_linewidth(1.2)
ax_c.set_xticks([0, 1]); ax_c.set_xticklabels(["CC", "AD"], fontsize=10)
ax_c.set_ylabel("Mean FC (Pearson r)", fontsize=9); ax_c.tick_params(labelsize=7)
ax_c.set_title("(C) Mean FC\nper Session", fontsize=11, fontweight="bold")

# ── D & E: Example FC matrices (CC and AD) ──────────────────────────────
for col_i, (group_idx, gname, gcol) in enumerate([
        (ctrl_indices[0], "CC", "#2196F3"),
        (ad_indices[0],   "AD", "#E91E63")]):
    ax_d = fig.add_subplot(gs[1, col_i])
    fc   = FC_collected[group_idx][:100, :100]
    fc_s = fc[np.ix_(sorted_idx, sorted_idx)]
    im   = ax_d.imshow(fc_s, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
    plt.colorbar(im, ax=ax_d, shrink=0.75)
    for b in net_bounds:
        ax_d.axhline(b, color="white", lw=0.6)
        ax_d.axvline(b, color="white", lw=0.6)
    ax_d.set_xticks([]); ax_d.set_yticks([])
    ax_d.set_title(f"({'DE'[col_i]}) FC — {gname}",
                   fontsize=11, fontweight="bold", color=gcol)

# ── F: Population PCA scree ──────────────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 2])
ax_e.bar(range(1, 21), expl_var[:20] * 100, color="steelblue", edgecolor="white")
ax_e.set_xlabel("PC", fontsize=9); ax_e.set_ylabel("Explained var (%)", fontsize=9)
ax_e.set_title("(F) Population PCA\nScree", fontsize=11, fontweight="bold")
ax_e.tick_params(labelsize=7)

# ── G: FC-corr distribution CC vs AD ─────────────────────────────────────
ax_f = fig.add_subplot(gs[1, 3])
# FC-corr to the group mean FC
fc_ctrl_flat = fc_ctrl_mean[np.triu_indices(N_SITES, k=1)]
cc_vals_cc = [np.corrcoef(FC_collected[i][np.triu_indices(N_SITES, k=1)], fc_ctrl_flat)[0, 1]
              for i in ctrl_indices]
cc_vals_ad = [np.corrcoef(FC_collected[i][np.triu_indices(N_SITES, k=1)], fc_ctrl_flat)[0, 1]
              for i in ad_indices]
bins_f = np.linspace(min(min(cc_vals_cc), min(cc_vals_ad)),
                     max(max(cc_vals_cc), max(cc_vals_ad)), 25)
ax_f.hist(cc_vals_cc, bins=bins_f, alpha=0.65, color="#2196F3",
          label=f"CC (med={np.median(cc_vals_cc):.3f})")
ax_f.hist(cc_vals_ad, bins=bins_f, alpha=0.65, color="#E91E63",
          label=f"AD (med={np.median(cc_vals_ad):.3f})")
ax_f.set_xlabel("FC-corr to CC mean", fontsize=9)
ax_f.set_ylabel("Count", fontsize=9)
ax_f.legend(fontsize=8); ax_f.tick_params(labelsize=7)
ax_f.set_title("(G) FC similarity\nto CC mean", fontsize=11, fontweight="bold")

# ── Network legend ────────────────────────────────────────────────────────
if have_atlas:
    handles = [Patch(color=c, label=n) for n, c in NET_COLORS_MAP.items()]
    fig.legend(handles=handles, loc="lower center", ncol=7, fontsize=8,
               bbox_to_anchor=(0.5, 0.0), title="Schaefer-100 networks",
               title_fontsize=8)

fig.savefig(f"{OUT_DIR}/Fig1_dataset.png", dpi=150, bbox_inches="tight")
plt.close()
print("Fig1 saved")

Fig1 saved


In [7]:
import sys; sys.path.insert(0, ".")
from res import RESERVOIRE_SIMPLE
from tqdm import trange

N_HIDDEN      = 2000
SPECTRAL_RAD  = 0.95
RECURRENT_FAC = 0.1
NOISE_RES     = 0.025
TSKIP_RES     = 10
MAX_DELAY     = 20

par = dict(tau_m_f=0.0005, tau_m_s=0.0005, N=N_HIDDEN,
           T=trial_duration, dt=0.005, sigma_input=0.01,
           shape=(N_HIDDEN, N_SITES, N_SITES, trial_duration))
res_nb = RESERVOIRE_SIMPLE(par)
sr     = max(abs(np.linalg.eigvals(res_nb.J)))
res_nb.J *= SPECTRAL_RAD / sr

rng_r      = np.random.default_rng(RNG_SEED)
Y_emp_list = []     # PC-projected empirical signal, (N_sites, T) per subject
Y_sim_list = []     # closed-loop simulated signal,  (N_sites, T-1) per subject

for idx in trange(len(concatenated_signals_list), desc="Reservoir fit+sim"):
    s   = concatenated_signals_list[idx]          # (N_sites, T)
    T_s = s.shape[1]
    res_nb.T = T_s; res_nb.reset()

    # PC projection (same as original notebook Cell 22)
    target = (s.T @ ev50 @ ev50.T).T              # (N_sites, T)
    Y_emp_list.append(target)

    # Teacher-forced pass
    X_raw = []
    for t in range(T_s - 1):
        res_nb.step_rate(RECURRENT_FAC * target[:, t], sigma_dyn=0.)
        X_raw.append(res_nb.X.copy())
    X_fit = np.array(X_raw)[TSKIP_RES:]           # (T-1-TSKIP, N_hidden)
    Y_fit = target[:, TSKIP_RES:TSKIP_RES + len(X_fit)].T   # same rows, (rows, N_sites)
    noise = rng_r.normal(0, NOISE_RES, X_fit.shape)
    W_out = np.linalg.pinv(X_fit + noise) @ Y_fit  # (N_hidden, N_sites)

    # Closed-loop test run — keep state from teacher-forced pass (no reset!)
    # Without this the reservoir starts at zero and produces near-zero output.
    res_nb.Jout = W_out.T.copy()                   # (N_sites, N_hidden)
    res_nb.y    = res_nb.Jout @ res_nb.X           # update readout for new weights
    Y_sim = []
    for t in range(T_s - 1):
        res_nb.step_rate(RECURRENT_FAC * res_nb.y, sigma_dyn=0.)
        Y_sim.append(res_nb.y.copy())
    Y_sim_list.append(np.array(Y_sim).T)           # (N_sites, T-1)

print(f"Done.  Y_emp[0]: {Y_emp_list[0].shape}   Y_sim[0]: {Y_sim_list[0].shape}")

# ── Delayed FC correlation per subject ────────────────────────────────────────
def compute_delayed_fc(data, delay):
    if delay == 0:
        return np.nan_to_num(np.corrcoef(data))
    lead = data[:, :-delay]; lag = data[:, delay:]
    C = np.corrcoef(lead, lag)
    return np.nan_to_num(C[data.shape[0]:, :data.shape[0]])

all_r_es, all_r_ee, all_r_tr = [], [], []   # emp-sim, emp-self, test-retest

for i in range(len(Y_emp_list)):
    emp  = Y_emp_list[i][:, TSKIP_RES:]           # skip transient
    sim  = Y_sim_list[i][:, TSKIP_RES:]
    Tmin = min(emp.shape[1], sim.shape[1])
    emp  = emp[:, :Tmin];  sim = sim[:, :Tmin]
    emp_e = emp[:, ::2];   emp_o = emp[:, 1::2]   # even / odd for test-retest

    fc0 = compute_delayed_fc(emp, 0)
    r_es, r_ee, r_tr = [], [], []
    for d in range(MAX_DELAY + 1):
        if emp.shape[1] <= d or emp_e.shape[1] <= d:
            r_es.append(np.nan); r_ee.append(np.nan); r_tr.append(np.nan); continue
        fc_e = compute_delayed_fc(emp,   d)
        fc_s = compute_delayed_fc(sim,   d)
        fc_v = compute_delayed_fc(emp_e, d)
        fc_o = compute_delayed_fc(emp_o, d)
        r_es.append(np.corrcoef(fc_e.flat, fc_s.flat)[0, 1])
        r_ee.append(np.corrcoef(fc_e.flat, fc0.flat)[0, 1])
        r_tr.append(np.corrcoef(fc_v.flat, fc_o.flat)[0, 1])
    all_r_es.append(r_es); all_r_ee.append(r_ee); all_r_tr.append(r_tr)

all_r_es = np.array(all_r_es)   # (N_subj, MAX_DELAY+1)
all_r_ee = np.array(all_r_ee)
all_r_tr = np.array(all_r_tr)
print(f"Delay arrays: {all_r_es.shape}")


Reservoir fit+sim:   0%|          | 0/185 [00:00<?, ?it/s]

Reservoir fit+sim:   1%|          | 1/185 [00:00<00:40,  4.54it/s]

Reservoir fit+sim:   1%|          | 2/185 [00:00<00:41,  4.39it/s]

Reservoir fit+sim:   2%|▏         | 3/185 [00:00<00:41,  4.33it/s]

Reservoir fit+sim:   2%|▏         | 4/185 [00:00<00:41,  4.41it/s]

...

Reservoir fit+sim:  99%|█████████▉| 184/185 [00:43<00:00,  4.22it/s]

Reservoir fit+sim: 100%|██████████| 185/185 [00:43<00:00,  4.29it/s]

Reservoir fit+sim: 100%|██████████| 185/185 [00:43<00:00,  4.21it/s]

Done.  Y_emp[0]: (121, 140)   Y_sim[0]: (121, 139)


Delay arrays: (185, 21)


In [8]:
import matplotlib as _mpl
_mpl.rcParams.update({"axes.labelsize": 9, "xtick.labelsize": 8,
                      "ytick.labelsize": 8, "legend.fontsize": 8,
                      "axes.titlesize": 10, "font.family": "sans-serif"})

def _tag(ax, letter, x=-0.14, y=1.06):
    ax.text(x, y, letter, transform=ax.transAxes,
            fontsize=13, fontweight="bold", va="top", ha="left")

def _clean(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig2 = plt.figure(figsize=(14, 13), facecolor="white")
gs2  = gridspec.GridSpec(3, 3, figure=fig2,
                         hspace=0.40, wspace=0.38,
                         top=0.97, bottom=0.04,
                         height_ratios=[1, 1, 0.72])

# ── A: LDA accuracy vs k (loaded from project root) ──────────────────────
ax_a = fig2.add_subplot(gs2[0, :2])
lda_path = os.path.join(ROOT, "fig_lda.png")
if os.path.exists(lda_path):
    ax_a.imshow(mpimg.imread(lda_path))
    ax_a.set_title("LDA accuracy vs. $k$ archetypes", pad=3)
else:
    ax_a.text(0.5, 0.5, "fig_lda.png not found\n(run original notebook first)",
              ha="center", va="center", fontsize=9, color="red", transform=ax_a.transAxes)
ax_a.axis("off")
_tag(ax_a, "A")

# ── B: FC similarity to CC mean (violin) ─────────────────────────────────
ax_b = fig2.add_subplot(gs2[0, 2])
cc_cc = [np.corrcoef(FC_collected[i][np.triu_indices(N_SITES, k=1)],
                     fc_ctrl_flat)[0, 1] for i in ctrl_indices]
cc_ad = [np.corrcoef(FC_collected[i][np.triu_indices(N_SITES, k=1)],
                     fc_ctrl_flat)[0, 1] for i in ad_indices]
vp = ax_b.violinplot([cc_cc, cc_ad], positions=[0, 1], showmedians=True)
vp["bodies"][0].set_facecolor("#2196F3"); vp["bodies"][1].set_facecolor("#E91E63")
for kv in ["cbars", "cmins", "cmaxes", "cmedians"]:
    vp[kv].set_color("k"); vp[kv].set_linewidth(1.2)
ax_b.set_xticks([0, 1]); ax_b.set_xticklabels(["CC", "AD"])
ax_b.set_ylabel("FC corr. to CC mean")
ax_b.set_title("FC similarity to CC mean", pad=3)
_clean(ax_b); _tag(ax_b, "B")

# ── C: FC-corr distribution (histogram) ──────────────────────────────────
ax_c = fig2.add_subplot(gs2[1, 0])
allv   = cc_cc + cc_ad
bins_c = np.linspace(min(allv) - 0.02, max(allv) + 0.02, 28)
ax_c.hist(cc_cc, bins=bins_c, alpha=0.65, color="#2196F3",
          label=f"CC  (n={len(cc_cc)}, med={np.median(cc_cc):.3f})")
ax_c.hist(cc_ad, bins=bins_c, alpha=0.65, color="#E91E63",
          label=f"AD  (n={len(cc_ad)}, med={np.median(cc_ad):.3f})")
ax_c.axvline(np.median(cc_cc), color="#2196F3", lw=1.5, ls="--")
ax_c.axvline(np.median(cc_ad), color="#E91E63", lw=1.5, ls="--")
ax_c.set_xlabel("FC corr. to CC mean"); ax_c.set_ylabel("Count")
ax_c.set_title("FC distribution: CC vs. AD", pad=3)
ax_c.legend(frameon=False)
_clean(ax_c); _tag(ax_c, "C")

# ── D: Population PCA scree ──────────────────────────────────────────────
ax_d = fig2.add_subplot(gs2[1, 1])
ax_d.bar(range(1, 21), expl_var[:20] * 100, color="#455A64", edgecolor="white", lw=0.5)
ax_d.set_xlabel("Principal component"); ax_d.set_ylabel("Explained variance (%)")
ax_d.set_title("Population PCA scree", pad=3)
ax_d.set_xticks([1, 5, 10, 15, 20])
_clean(ax_d); _tag(ax_d, "D")

# ── E: Mean FC matrix (CC group) ─────────────────────────────────────────
ax_e = fig2.add_subplot(gs2[1, 2])
fc_s  = fc_ctrl_mean[:100, :100][np.ix_(sorted_idx, sorted_idx)]
im_e  = ax_e.imshow(fc_s, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
cb    = plt.colorbar(im_e, ax=ax_e, shrink=0.82, pad=0.03, fraction=0.046)
cb.set_label("Pearson r", fontsize=8); cb.ax.tick_params(labelsize=7)
for b in net_bounds:
    ax_e.axhline(b, color="white", lw=0.5); ax_e.axvline(b, color="white", lw=0.5)
ax_e.set_xticks([]); ax_e.set_yticks([])
ax_e.set_xlabel("Brain region"); ax_e.set_ylabel("Brain region")
ax_e.set_title("Mean FC — CC group", pad=3)
_tag(ax_e, "E")

# ── F: Temporal correlation decay ────────────────────────────────────────
ax_f = fig2.add_subplot(gs2[2, :])
delays_s = np.arange(MAX_DELAY + 1) * TR
_dec = [
    (all_r_es, "#1565C0", "Empirical vs. simulated FC"),
    (all_r_ee, "#757575", "Empirical FC vs. zero-lag FC"),
    (all_r_tr, "#212121", "Test-retest (even / odd halves, ceiling)"),
]
for arr, col, lbl in _dec:
    med = np.nanmedian(arr, axis=0)
    p25 = np.nanpercentile(arr, 25, axis=0)
    p75 = np.nanpercentile(arr, 75, axis=0)
    ax_f.plot(delays_s, med, color=col, label=lbl, lw=2)
    ax_f.fill_between(delays_s, p25, p75, color=col, alpha=0.18)
ax_f.axhline(0, color="k", ls=":", lw=0.8)
ax_f.set_xlabel("Delay (s)"); ax_f.set_ylabel("Pearson r  (delayed FC)")
ax_f.set_title("Temporal decay of functional connectivity", pad=3)
ax_f.set_ylim(-0.15, 1.05); ax_f.set_xlim(0, MAX_DELAY * TR)
ax_f.set_xticks(np.arange(0, MAX_DELAY * TR + 1, TR * 5))
ax_f.legend(frameon=True, framealpha=0.9, edgecolor="#cccccc", loc="upper right")
_clean(ax_f); _tag(ax_f, "F", x=-0.04)

fig2.savefig(f"{OUT_DIR}/Fig2_model.png", dpi=200, bbox_inches="tight")
plt.close()
print("Fig2 saved")

Fig2 saved


In [9]:
# All 6 panels load pre-computed PNGs from the project root.
# These were saved by the original Fig1DEF_MC_data.ipynb run.

png_layout = [
    # (row, col, filename,                    panel_label)
    (0, 0, "therapeutic_lda_kde.png",
           "(A) Offline interpolation — LDA score distributions per dose"),
    (0, 1, "therapeutic_doseresponse.png",
           "(B) Offline interpolation — Dose-response (3 metrics)"),
    (1, 0, "pert_lda_kde.png",
           "(C) Single-site perturbation — LDA score distributions per dose"),
    (1, 1, "pert_doseresponse.png",
           "(D) Single-site perturbation — Dose-response (3 metrics)"),
    (2, 0, "pert_per_patient.png",
           "(E) Single-site perturbation — Per-patient LDA trajectories"),
    (2, 1, "pert_site_importance.png",
           "(F) Single-site perturbation — Site importance"),
]

fig3, axes3 = plt.subplots(3, 2, figsize=(18, 18), facecolor="white")
fig3.suptitle("Figure 3 — Perturbation Experiments (AD → CC)",
              fontsize=15, fontweight="bold", y=1.001)

missing = []
for (r, c, fname, title) in png_layout:
    ax = axes3[r, c]
    full_path = os.path.join(ROOT, fname)
    if os.path.exists(full_path):
        img = mpimg.imread(full_path)
        ax.imshow(img)
        ax.set_title(title, fontsize=10, fontweight="bold", pad=4)
    else:
        ax.text(0.5, 0.5, f"{fname}\nnot found\n(run original notebook first)",
                ha="center", va="center", fontsize=10, color="red",
                transform=ax.transAxes)
        ax.set_title(title, fontsize=10, fontweight="bold", pad=4)
        missing.append(fname)
    ax.axis("off")

plt.tight_layout()
fig3.savefig(f"{OUT_DIR}/Fig3_perturbation.png", dpi=150, bbox_inches="tight")
plt.close()

if missing:
    print(f"WARNING: {len(missing)} PNG(s) not found: {missing}")
else:
    print("Fig3 saved — all 6 panels loaded from project root")
print("\nAll done! Check summary_out/ for the 3 figures.")

Fig3 saved — all 6 panels loaded from project root

All done! Check summary_out/ for the 3 figures.
